In [ ]:
import matplotlib.pyplot as plt
import os
import pickle
from contextlib import nullcontext
import torch
from model import GPTConfig, GPT
from torch.nn import functional as F
import math

In [ ]:
# Edit this to work for your format.
# This reduces the number of tokens generated, making analyzing the attention patterns a bit easier.
def result_complete(text):
    """Given a string containing the generated text, return True if the end text is found within the text."""
    # Recommendedation: Start by looking at the first copy to see if you can find a copy head
    return 'Evaluating: (' in text

    # Then look at the full text to see if you can find other interesting heads
    # return 'Final Result' in text and '\n' in text[text.find('Final Result'):]

In [ ]:
# Also modify the start expression to match your format.
out_dir = 'out-shakespeare-char' # ignored if init_from is not 'resume'
start = "Expression: (7 * (2 + 4))" # "\n" # or "<|endoftext|>" or etc. Can also specify a file, use as: "FILE:prompt.txt"

In [ ]:
"""
Sample from a trained model
"""
# -----------------------------------------------------------------------------
init_from = 'resume' # either 'resume' (from an out_dir) or a gpt2 variant (e.g. 'gpt2-xl')
num_samples = 1 # number of samples to draw
max_new_tokens = 500 # number of tokens generated in each sample
temperature = 0.0000008 # 1.0 = no change, < 1.0 = less random, > 1.0 = more random, in predictions
top_k = 200 # retain only the top_k most likely tokens, clamp others to have 0 probability
seed = 1337
device = 'cuda' # examples: 'cpu', 'cuda', 'cuda:0', 'cuda:1', etc.
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32' or 'bfloat16' or 'float16'
compile = False # use PyTorch 2.0 to compile the model to be faster
#---------------------
# init from a model saved in a specific directory
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cuda.matmul.allow_tf32 = True # allow tf32 on matmul
torch.backends.cudnn.allow_tf32 = True # allow tf32 on cudnn
device_type = 'cuda' if 'cuda' in device else 'cpu' # for later use in torch.autocast
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

ckpt_path = os.path.join(out_dir, 'ckpt.pt')
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
model = GPT(gptconf)
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
model.load_state_dict(state_dict)

model.eval()
model.to(device)
if compile:
    model = torch.compile(model) # requires PyTorch 2.0 (optional)

# look for the meta pickle in case it is available in the dataset folder
load_meta = False
if init_from == 'resume' and 'config' in checkpoint and 'dataset' in checkpoint['config']: # older checkpoints might not have these...
    meta_path = os.path.join('data', checkpoint['config']['dataset'], 'meta.pkl')
    load_meta = os.path.exists(meta_path)
if load_meta:
    print(f"Loading meta from {meta_path}...")
    with open(meta_path, 'rb') as f:
        meta = pickle.load(f)
    stoi, itos = meta['stoi'], meta['itos']
    encode = lambda s: [stoi[c] for c in s]
    decode = lambda l: ''.join([itos[i] for i in l])
else:
    raise Exception('This script can only load from a path')

start_ids = encode(start)
x = (torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...])

In [ ]:
text = decode(x[0].tolist())
text

In [ ]:
# This is pulled out of the nanoGPT source code.
# The line att = att.masked_fill(torch.tril(torch.ones(T, T, device=att.device)) == 0, float('-inf')) was a little painful to work out,
# so I'm just providing it to you.  Sorry if you used something other than nanoGPT! You'll need to extract your attention patterns in a different way!
def generate_print_attn(self, idx, max_new_tokens, temperature=1.0, top_k=None):
    """
    Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
    the sequence max_new_tokens times, feeding the predictions back into the model each time.
    Most likely you'll want to make sure to be in model.eval() mode of operation for this.
    """
    for _ in range(max_new_tokens):
        atts = []
        # if the sequence context is growing too long we must crop it at block_size
        idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
        # forward the model to get the logits for the index in the sequence

        ## NEW VERSION
        b, t = idx_cond.size()
        pos = torch.arange(0, t, dtype=torch.long, device=device)
        tok_emb = self.transformer.wte(idx_cond)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        ## END NEW VERSION

        for block in self.transformer.h:
            old_x = x
            x = block.ln_1(x)
            B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

            # calculate query, key, values for all heads in batch and move head forward to be the batch dim
            q, k, v  = block.attn.c_attn(x).split(block.attn.n_embd, dim=2)
            k = k.view(B, T, block.attn.n_head, C // block.attn.n_head).transpose(1, 2) # (B, nh, T, hs)
            q = q.view(B, T, block.attn.n_head, C // block.attn.n_head).transpose(1, 2) # (B, nh, T, hs)
            v = v.view(B, T, block.attn.n_head, C // block.attn.n_head).transpose(1, 2) # (B, nh, T, hs)

            # # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
            # if block.attn.flash: # This is what nanoGPT uses by default. But we avoid it here to materialize the attention matrices.
            #     # efficient attention using Flash Attention CUDA kernels
            #     y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if block.attn.training else 0, is_causal=True)
            # else:
            # manual implementation of attention
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(torch.tril(torch.ones(T, T, device=att.device)) == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = block.attn.attn_dropout(att)
            atts.append(att)
            # print(att)
            y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
            y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

            # output projection
            y = block.attn.resid_dropout(block.attn.c_proj(y))
            x = old_x + y
            x = x + block.mlp(block.ln_2(x))

        ## NEW VERSION
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        ## END NEW VERSION

        # pluck the logits at the final step and scale by desired temperature
        logits = logits[:, -1, :] / temperature
        # optionally crop the logits to only the top k options
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')
        # apply softmax to convert logits to (normalized) probabilities
        probs = F.softmax(logits, dim=-1)
        # sample from the distribution
        idx_next = torch.multinomial(probs, num_samples=1)
        # append sampled index to the running sequence and continue
        idx = torch.cat((idx, idx_next), dim=1)

        text = decode(idx[0].tolist())
        if result_complete(text):
            break

    return idx, atts
 

In [ ]:
# run generation
with torch.no_grad():
    with ctx:
        for k in range(num_samples):
            y,atts = generate_print_attn(model,x, max_new_tokens, temperature=temperature, top_k=top_k)
            print(decode(y[0].tolist()))
            print('-------SAMPLE ENDS HERE SO FAR--------')
            # if 'Evaluating: (' in decode(y[0].tolist()):
            #     break


In [ ]:
len(atts)

In [ ]:
text = decode(y[0].tolist())
out_text = text
# plt.figure(figsize=(15,15)) # If you have a lot of tokens, this will let you zoom in and see the matrix a little closer
plt.imshow(atts[3].cpu()[0,5,:,:])
plt.xticks([x for x in range(0,len(out_text))], out_text)
plt.yticks([x for x in range(0,len(out_text))], out_text)
plt.show()

In [ ]:
out_text = text
# plt.figure(figsize=(15,15))
plt.imshow(atts[5].cpu()[0,0,:,:])
plt.xticks([x for x in range(0,len(out_text))], out_text)
plt.yticks([x for x in range(0,len(out_text))], out_text)
plt.show()

In [ ]:
fig, axs = plt.subplots(len(atts), atts[0].cpu().shape[1],figsize=(15,15))
for i in range(0,len(atts)):
    for j in range(0,atts[i].cpu().shape[1]):
        axs[i,j].imshow(atts[i].cpu()[0,j,:,:])

In [ ]:
# TODO: Copy a few interesting attention head matrices down here
# Discuss what you think each head might be doing.